## Convert a regular LLM model into a full-fledged DeepSeek-R1 like reasoning model 
Finetuned using DeekSeek's GRPO


GRPO can magically create the reasoning process for standard LLM model. it's Renforce Learning (RL) algorithm that optimizes responses efficiently without requiring a value function.

Reenforce learning training such as Proximal Policy Optimization (PPO) which relies on a value function and had to collect large swathes of data to fill the working out / chain of thought process. 

here's how GRPO works:

    The model generates groups of responses
    Each response is scored based on correctness or another metric created by some set reward function rather than an LLM reward model.
    The average score of the group is computed.
    Each response's score is compared to the group average.
    The model is reinforced to favor higher-scoring responses.


When DeepSeek’s researchers training R1-Zero with pure reinforcement learning (RL) discovered the model learned to extend its thinking time by reevaluating its initial approach, without any human guidance or predefined instructions.

let's convert a regular model into reasoning model which reproduce Deepseek R1-Zero's thinking model on this notebook.

#### Notebook Samples

In [ ]:
# So let's an example first BEFORE and AFTER

BEFORE (Qwen2.5:3b no reasoning)
++++++++++++++++++++++++++++++
❯ ollama run qwen2.5:3b
>>> How many r's are in strawberry?
There is one "r" in the word "strawberry."

AFTER (Qwen2.5:3b with GRPO)
++++++++++++++++++++++++++++++
❯ ollama run qwen2.5-3b-think-rl
>>> How many r's are in strawberry?
<thinking>
To find out how many times 'r' appears in "strawberry", I'll go through the word character by character 
and count each occurrence.
The word is: strawberry
I see:
- The letter 'r' starts at index 2, so that's one.
- It repeats at indices 4 and 7, so that's two more.
Therefore, there are three occurrences of the letter 'r'.
</thinking>

<answer>3</answer>

#### Ollama Samples

BEFORE

❯ ollama run qwen2.5:3b
>>> Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did s
... he earn?
To calculate Weng's earnings for the 50 minutes of babysitting, we need to follow these steps:

1. Determine how many hours 50 minutes is equivalent to.
2. Calculate her earnings based on that number of hours.

First, let's convert 50 minutes into hours:
\[ \text{Hours} = \frac{\text{Minutes}}{60} = \frac{50}{60} = \frac{5}{6} \text{ hours} \]

Next, calculate her earnings for that duration. Given that she earns $12 per hour:
\[ \text{Earnings} = \left(\frac{5}{6}\right) \times 12 = 5 \text{ dollars} \]

So, Weng earned **$5** from babysitting for 50 minutes.


AFTER
❯ ollama run mychen76/qwen2.5-3b-think-r1
>>> Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did s
... he earn?
<thinking>
To find out how much Weng earned from her babysitting yesterday, we need to calculate the earnings 
based on the rate and the time worked.
Firstly, convert the working time (50 minutes) into hours because the wage is given per hour. There are 
60 minutes in an hour, so:
\[ \text{Time in Hours} = \frac{50}{6mlt} = 8.333... \text{hours} \]
Since Weng earns $12 for every hour she babysits, her earnings can be calculated as follows:
\[ \text{Earnings} = 12 \times 0.833\ldots = $10 \text{ (approximately)} \]
</thinking>
<answer>$10</answer>



### Prepare Environment

#### Install uv
https://docs.astral.sh/uv/getting-started/installation/
##### linux: 
"curl -LsSf https://astral.sh/uv/install.sh | sh"

##### create new environment
uv venv myenv --python 3.12 --seed
source myenv/bin/activate

##### Install vLLM with CUDA 12.1.
uv pip install vllm 
uv pip install unsloth
uv pip install --upgrade pillow
uv pip install diffusers
uv pip install git+https://github.com/huggingface/trl.git@e95f9fb74a3c3647b86f251b7e230ec51c64b72b

##### jupyterlab
uv pip install jupyterlab
uv pip install -U ipywidgets

In [1]:
import torch
def check_pytorch_gpu():
    try:
        if torch.cuda.is_available():
            print(f"PyTorch can access {torch.cuda.device_count()} GPU(s).")
            for i in range(torch.cuda.device_count()):
                print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
        else:
            print("PyTorch cannot access any GPUs.")
    except Exception as e:
        print(f"An error occurred: {e}")

if __name__ == "__main__":
    check_pytorch_gpu()

PyTorch can access 1 GPU(s).
GPU 0: NVIDIA GeForce RTX 3090


### Unsloth Training Setup

Use `PatchFastRL` before all functions to patch GRPO and other RL algorithms!

In [2]:
from unsloth import FastLanguageModel, PatchFastRL
PatchFastRL("GRPO", FastLanguageModel)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 02-11 17:00:45 __init__.py:190] Automatically detected platform cuda.


Load up `Qwen 2.5 3B Instruct`, and set parameters

In [4]:
from unsloth import is_bfloat16_supported
import torch
max_seq_length = 1024 # Can increase for longer reasoning traces
lora_rank = 64 # Larger rank = smarter, but slower

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-3B-Instruct",
    max_seq_length = max_seq_length,
    load_in_4bit = True, # False for LoRA 16bit
    fast_inference = True, # Enable vLLM fast inference
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.5, # Reduce if out of memory
)

model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ], # Remove QKVO if out of memory
    lora_alpha = lora_rank,
    use_gradient_checkpointing = "unsloth", # Enable long context finetuning
    random_state = 3407,
)

==((====))==  Unsloth 2025.2.5: Fast Qwen2 patching. Transformers: 4.48.2.
   \\   /|    GPU: NVIDIA GeForce RTX 3090. Max memory: 23.58 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 8.6. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.28.post3. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit with actual GPU utilization = 49.29%
Unsloth: Your GPU has CUDA compute capability 8.6 with VRAM = 23.58 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 1024. Num Sequences = 224.
Unsloth: vLLM's KV Cache can use up to 9.2 GB. Also swap space = 5 GB.
INFO 02-11 17:00:53 config.py:542] This model supports multiple tasks: {'generate', 'reward', 'score', 'classify', 'embed'}. Defaulting to 'generate'.
Unsloth: vLLM Bitsandbytes config using kwar

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 02-11 17:00:55 model_runner.py:1115] Loading model weights took 2.2265 GB
INFO 02-11 17:00:55 punica_selector.py:18] Using PunicaWrapperGPU.
INFO 02-11 17:00:56 worker.py:267] Memory profiling takes 0.92 seconds
INFO 02-11 17:00:56 worker.py:267] the current vLLM instance can use total_gpu_memory (23.58GiB) x gpu_memory_utilization (0.49) = 11.62GiB
INFO 02-11 17:00:56 worker.py:267] model weights take 2.23GiB; non_torch_memory takes 0.06GiB; PyTorch activation peak memory takes 1.22GiB; the rest of the memory reserved for KV Cache is 8.12GiB.
INFO 02-11 17:00:56 executor_base.py:110] # CUDA blocks: 14782, # CPU blocks: 9102
INFO 02-11 17:00:56 executor_base.py:115] Maximum concurrency for 1024 tokens per request: 230.97x
INFO 02-11 17:01:01 model_runner.py:1434] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If out-of-memory error 

Capturing CUDA graph shapes: 100%|██████████████████████████████████████████████████████████████████| 31/31 [00:17<00:00,  1.82it/s]

INFO 02-11 17:01:18 model_runner.py:1562] Graph capturing finished in 17 secs, took 0.73 GiB
INFO 02-11 17:01:18 llm_engine.py:431] init engine (profile, create kv cache, warmup model) took 23.26 seconds



Unsloth 2025.2.5 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


### Preparate Training Data
<a name="Data"></a>

In [5]:
import re
from datasets import load_dataset, Dataset

data = load_dataset('openai/gsm8k', 'main')["train"]
data

Dataset({
    features: ['question', 'answer'],
    num_rows: 7473
})

##### Sample Raw Data

```
question:
Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?

answer:
Weng earns 12/60 = $<<12/60=0.2>>0.2 per minute. Working 50 minutes, she earned 0.2 x 50 = $<<0.2*50=10>>10. #### 10
```
###10

map this to training dataset

Weng earns 12/60 = $<<12/60=0.2>>0.2 per minute. Working 50 minutes, she earned 0.2 x 50 = $<<0.2*50=10>>10.


In [30]:
import re
from datasets import load_dataset, Dataset

# Load and prep dataset
SYSTEM_PROMPT = """
Respond in the following format:
<thinking>
...
</thinking>
<answer>
...
</answer>
"""

def extract_hash_answer(text: str) -> str | None:
    if "####" not in text:
        return None
    return text.split("####")[1].strip()

# uncomment middle messages for 1-shot prompting
def get_gsm8k_questions(split = "train") -> Dataset:
    data = load_dataset('openai/gsm8k', 'main')[split] # type: ignore
    data = data.map(lambda x: { # type: ignore
        'prompt': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': x['question']}
        ],
        'answer': extract_hash_answer(x['answer'])
    }) # type: ignore
    return data # type: ignore

dataset = get_gsm8k_questions()
dataset

Dataset({
    features: ['question', 'answer', 'prompt'],
    num_rows: 7473
})

In [36]:
dataset["question"][1]
dataset["answer"][1]
dataset["prompt"][1]

[{'content': '\nRespond in the following format:\n<thinking>\n...\n</thinking>\n<answer>\n...\n</answer>\n',
  'role': 'system'},
 {'content': 'Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?',
  'role': 'user'}]

## Reward Functions

In [6]:
import re

XML_COT_FORMAT = """\
<thinking>
{thinking}
</thinking>
<answer>
{answer}
</answer>
"""

def extract_xml_answer(text: str) -> str:
    answer = text.split("<answer>")[-1]
    answer = answer.split("</answer>")[0]
    return answer.strip()

# Reward functions
def correctness_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    responses = [completion[0]['content'] for completion in completions]
    q = prompts[0][-1]['content']
    extracted_responses = [extract_xml_answer(r) for r in responses]
    #print('-'*20, f"Question:\n{q}", f"\nAnswer:\n{answer[0]}", f"\nResponse:\n{responses[0]}", f"\nExtracted:\n{extracted_responses[0]}")
    return [2.0 if r == a else 0.0 for r, a in zip(extracted_responses, answer)]

def int_reward_func(completions, **kwargs) -> list[float]:
    responses = [completion[0]['content'] for completion in completions]
    extracted_responses = [extract_xml_answer(r) for r in responses]
    return [0.5 if r.isdigit() else 0.0 for r in extracted_responses]

def strict_format_reward_func(completions, **kwargs) -> list[float]:
    """Reward function that checks if the completion has a specific format."""
    pattern = r"^<thinking>\n.*?\n</thinking>\n<answer>\n.*?\n</answer>\n$"
    responses = [completion[0]["content"] for completion in completions]
    matches = [re.match(pattern, r) for r in responses]
    return [0.5 if match else 0.0 for match in matches]

def soft_format_reward_func(completions, **kwargs) -> list[float]:
    """Reward function that checks if the completion has a specific format."""
    pattern = r"<thinking>.*?</thinking>\s*<answer>.*?</answer>"
    responses = [completion[0]["content"] for completion in completions]
    matches = [re.match(pattern, r) for r in responses]
    return [0.5 if match else 0.0 for match in matches]

def count_xml(text) -> float:
    count = 0.0
    if text.count("<thinking>\n") == 1:
        count += 0.125
    if text.count("\n</thinking>\n") == 1:
        count += 0.125
    if text.count("\n<answer>\n") == 1:
        count += 0.125
        count -= len(text.split("\n</answer>\n")[-1])*0.001
    if text.count("\n</answer>") == 1:
        count += 0.125
        count -= (len(text.split("\n</answer>")[-1]) - 1)*0.001
    return count

def xmlcount_reward_func(completions, **kwargs) -> list[float]:
    contents = [completion[0]["content"] for completion in completions]
    return [count_xml(c) for c in contents]

<a name="Train"></a>
### Train the model

Now set up GRPO Trainer and all configurations!

In [7]:
from trl import GRPOConfig, GRPOTrainer
training_args = GRPOConfig(
    use_vllm = True, # use vLLM for fast inference!
    learning_rate = 5e-6,
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.1,
    warmup_ratio = 0.1,
    lr_scheduler_type = "cosine",
    optim = "adamw_8bit",
    logging_steps = 1,
    bf16 = is_bfloat16_supported(),
    fp16 = not is_bfloat16_supported(),
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 1, # Increase to 4 for smoother training
    num_generations = 8, # Decrease if out of memory
    max_prompt_length = 256,
    max_completion_length = 200,
    #num_train_epochs = 1, # Set to 1 for a full training run
    max_steps = 250,
    save_steps = 250,
    max_grad_norm = 0.1,
    report_to = "none", # Can use Weights & Biases
    output_dir = "outputs",
)

torch.distributed process group is initialized, but parallel_mode != ParallelMode.DISTRIBUTED. In order to use Torch DDP, launch your script with `python -m torch.distributed.launch


And let's run the trainer! If you scroll up, you'll see a table of rewards. The goal is to see the `reward` column increase!

You might have to wait 150 to 200 steps for any action. You'll probably get 0 reward for the first 100 steps. Please be patient!

| Step | Training Loss | reward    | reward_std | completion_length | kl       |
|------|---------------|-----------|------------|-------------------|----------|
| 1    | 0.000000      | 0.125000  | 0.000000   | 200.000000        | 0.000000 |
| 2    | 0.000000      | 0.072375  | 0.248112   | 200.000000        | 0.000000 |
| 3    | 0.000000      | -0.079000 | 0.163776   | 182.500000        | 0.000005 |


In [ ]:
trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        xmlcount_reward_func,
        soft_format_reward_func,
        strict_format_reward_func,
        int_reward_func,
        correctness_reward_func,
    ],
    args = training_args,
    train_dataset = dataset,
)
trainer.train()

<a name="Inference"></a>
### Direct Inference
Now let's try the model we just trained! First, let's first try the model without any GRPO trained:

In [18]:
text = tokenizer.apply_chat_template([
    {"role" : "user", "content" : "How many r's are in strawberry?"},
], tokenize = False, add_generation_prompt = True)

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 0.8,
    top_p = 0.95,
    max_tokens = 1024,
)
output = model.fast_generate(
    [text],
    sampling_params = sampling_params,
    lora_request = None,
)[0].outputs[0].text

output

Processed prompts: 100%|█| 1/1 [00:00<00:00,  4.68it/s, est. speed input: 173.61 toks/s, output: 70.38 toks/s


'There are two \'r\'s in the word "strawberry".'

And now with the LoRA we just trained with GRPO - we first save the LoRA first!

In [16]:
model.save_lora("grpo_saved_lora")

## Restart Kernel

## Load LoRA Adapter Test

In [19]:
import torch
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "grpo_saved_lora", # Lora MODEL YOU USED FOR TRAINING
    max_seq_length = 1024,
    load_in_4bit = True,
    fast_inference = True,
    dtype = None,  # autodetect
    gpu_memory_utilization = 0.5
)
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

SYSTEM_PROMPT = """
Respond in the following format:
<thinking>
...
</thinking>
<answer>
...
</answer>
"""

==((====))==  Unsloth 2025.2.5: Fast Qwen2 patching. Transformers: 4.48.2.
   \\   /|    GPU: NVIDIA GeForce RTX 3090. Max memory: 23.58 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 8.6. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.28.post3. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit with actual GPU utilization = 15.55%
Unsloth: Your GPU has CUDA compute capability 8.6 with VRAM = 23.58 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 1024. Num Sequences = 128.
Unsloth: vLLM's KV Cache can use up to 1.25 GB. Also swap space = 5 GB.
INFO 02-11 22:22:06 config.py:542] This model supports multiple tasks: {'generate', 'reward', 'score', 'classify', 'embed'}. Defaulting to 'generate'.
Unsloth: vLLM Bitsandbytes config using kwa

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 02-11 22:22:11 model_runner.py:1115] Loading model weights took 2.2170 GB
INFO 02-11 22:22:14 worker.py:267] Memory profiling takes 1.73 seconds
INFO 02-11 22:22:14 worker.py:267] the current vLLM instance can use total_gpu_memory (23.58GiB) x gpu_memory_utilization (0.16) = 3.67GiB
INFO 02-11 22:22:14 worker.py:267] model weights take 2.22GiB; non_torch_memory takes 0.00GiB; PyTorch activation peak memory takes 0.69GiB; the rest of the memory reserved for KV Cache is 0.76GiB.
INFO 02-11 22:22:15 executor_base.py:110] # CUDA blocks: 1378, # CPU blocks: 9102
INFO 02-11 22:22:15 executor_base.py:115] Maximum concurrency for 1024 tokens per request: 21.53x
INFO 02-11 22:22:21 model_runner.py:1434] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If out-of-memory error occurs during cudagraph capture, consider decreasing `gpu_memory_utili

Capturing CUDA graph shapes: 100%|███████████████████████████████████████████| 19/19 [01:00<00:00,  3.17s/it]

INFO 02-11 22:23:21 model_runner.py:1562] Graph capturing finished in 60 secs, took 0.46 GiB
INFO 02-11 22:23:21 llm_engine.py:431] init engine (profile, create kv cache, warmup model) took 70.27 seconds


In [20]:
q1="How many r's are in strawberry?"

text = tokenizer.apply_chat_template([
    {"role" : "system", "content" : SYSTEM_PROMPT},
    {"role" : "user", "content" : q1},
], tokenize = False, add_generation_prompt = True)

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 0.8,
    top_p = 0.95,
    max_tokens = 1024,
)
output = model.fast_generate(
    text,
    sampling_params = sampling_params,
    lora_request = None,
)[0].outputs[0].text

print(output)

Processed prompts: 100%|█| 1/1 [00:00<00:00,  1.03it/s, est. speed input: 42.28 toks/s, output: 109.30 toks/s

<thinking>
To find out how many 'r's are in the word "strawberry", I will go through the word letter by letter and count the occurrences of the letter 'r'.
<thinking>
s
t
r
a
w
b
a
r
r
y
...
After going through the word, I find that the letter 'r' appears 3 times.
</thinking>
<answer>There are 3 r's in the word "strawberry".</answer>


In [22]:
q2="A family of 12 monkeys collected 10 piles of bananas. 6 piles had 9 hands, with each hand having 14 bananas, while the remaining piles had 12 hands, with each hand having 9 bananas. How many bananas would each monkey get if they divide the bananas equally amongst themselves?"

text = tokenizer.apply_chat_template([
    {"role" : "system", "content" : SYSTEM_PROMPT},
    {"role" : "user", "content" : q2},
], tokenize = False, add_generation_prompt = True)

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 0.8,
    top_p = 0.95,
    max_tokens = 1024,
)
output2 = model.fast_generate(
    text,
    sampling_params = sampling_params,
    lora_request = None #model.load_lora("grpo_saved_lora"),
)[0].outputs[0].text

print(output2)


Processed prompts: 100%|█| 1/1 [00:02<00:00,  2.20s/it, est. speed input: 44.94 toks/s, output: 110.76 toks/s

<thinking>
First, we need to calculate the total number of bananas collected by the monkeys. We know there are 6 piles with 9 hands each, with 14 bananas per hand, and 4 piles with 12 hands each, with 9 bananas per hand.

For the 6 piles with 9 hands each:
\[6 \times 9 \times 14 = 6 \times 126 = 756\]

For the 4 piles with 12 hands each:
\[4 \times 12 \times 9 = 4 \times 108 = 432\]

Now, we add these two totals to find the overall total of bananas:
\[756 + 432 = 1188\]

Next, we need to divide the total number of bananas by the number of monkeys (12) to find out how many bananas each monkey would get:
\[1188 \div 12 = 99\]

So, each monkey would get 99 bananas.
</thinking>
<answer>
Each monkey would get 99 bananas. </answer>


Our reasoning model is much better - it's not always correct, since we only trained it for an hour or so - it'll be better if we extend the sequence length and train for longer!